# 00 — Do XML da ANA à tabela pré-formatada (`data/raw`)

Este notebook mostra, passo a passo, como as respostas originais da ANA e do Porto de Manaus viram as tabelas de `data/raw/`. **Nenhum valor é alterado** nesse caminho: só muda o formato (XML → tabela, mês → dia) e o tipo (texto → número/data).

O código é o mesmo do pipeline ([`ana.py`](../src/supplyrisk/ana.py) e [`porto.py`](../src/supplyrisk/porto.py)); aqui cada etapa é chamada separadamente para dar para ver o que acontece.

| Etapa | Função | Saída |
| --- | --- | --- |
| 1. Resposta original | `xml_convencional` / `xml_telemetria` | XML (do cache `data/cache/ana/`) |
| 2. XML como tabela | `tabela_xml` | um registro por linha, tudo texto |
| 3. Pré-formatado | `ler_convencional` / `ler_telemetria` | tabela tipada, igual à de `data/raw/` |

Dicionário de dados: [`docs/dados.md`](../docs/dados.md).

In [1]:
import pandas as pd

from supplyrisk import ana, porto

pd.set_option("display.max_columns", 20)
MANAUS, MANACAPURU = 14990000, 14100000

---
## Série convencional (leitura da régua)

### 1. Resposta original

A ANA devolve um XML por consulta. Pedimos um ano por vez. Exemplo: Manaus, 2010.

In [2]:
xml = ana.xml_convencional(MANAUS, 2010)
texto = xml.decode("utf-8")
inicio = texto.index("<SerieHistorica ")
print(texto[inicio:inicio + 1500])

<SerieHistorica diffgr:id="SerieHistorica1" msdata:rowOrder="0">
        <EstacaoCodigo>14990000</EstacaoCodigo>
        <NivelConsistencia>1</NivelConsistencia>
        <DataHora>2010-12-01 00:00:00</DataHora>
        <MediaDiaria>1</MediaDiaria>
        <TipoMedicaoCotas>1</TipoMedicaoCotas>
        <Maxima>1861</Maxima>
        <Minima>1641</Minima>
        <Media>1771</Media>
        <DiaMaxima>31</DiaMaxima>
        <DiaMinima>1</DiaMinima>
        <MaximaStatus>0</MaximaStatus>
        <MinimaStatus>0</MinimaStatus>
        <MediaStatus>0</MediaStatus>
        <MediaAnualStatus>0</MediaAnualStatus>
        <Cota01>1645</Cota01>
        <Cota02>1660.5</Cota02>
        <Cota03>1673</Cota03>
        <Cota04>1685</Cota04>
        <Cota05>1699</Cota05>
        <Cota06>1714.5</Cota06>
        <Cota07>1721.5</Cota07>
        <Cota08>1732</Cota08>
        <Cota09>1742.5</Cota09>
        <Cota10>1751</Cota10>
        <Cota11>1757</Cota11>
        <Cota12>1761.5</Cota12>
        <Cota13>17

### 2. XML como tabela

Cada `<SerieHistorica>` é **um mês** de uma versão (bruta/consistida, média/leitura avulsa). Os dias estão em colunas: `Cota01` … `Cota31`, e a qualidade de cada dia em `Cota01Status` … `Cota31Status`. Tudo ainda é texto.

In [3]:
meses = ana.tabela_xml(xml, "SerieHistorica")
print(meses.shape, "→ (meses × versões, campos)")
meses.head(5)

(48, 78) → (meses × versões, campos)


,EstacaoCodigo,NivelConsistencia,DataHora,MediaDiaria,TipoMedicaoCotas,Maxima,Minima,Media,DiaMaxima,DiaMinima,...,Cota22Status,Cota23Status,Cota24Status,Cota25Status,Cota26Status,Cota27Status,Cota28Status,Cota29Status,Cota30Status,Cota31Status
0,14990000,1,2010-12-01 00:00:00,1,1,1861,1641,1771,31,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,14990000,2,2010-12-01 00:00:00,1,1,1860,1645,1770.9032,31,1,...,1,1,1,1,1,1,1,1,1,1
2,14990000,1,2010-12-01 07:00:00,0,1,1861,1641,1771,31,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,14990000,1,2010-12-01 17:00:00,0,1,1861,1641,1771,31,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,14990000,1,2010-11-01 00:00:00,1,1,1634,1397,1479,30,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0


In [4]:
# As versões do mesmo mês: consistência 1 (bruto) e 2 (consistido); média diária ou leitura das 7h/17h
meses[meses["DataHora"].str.startswith("2010-10")][
    ["NivelConsistencia", "DataHora", "MediaDiaria", "Cota01", "Cota01Status", "Cota24", "DataIns"]
]

,NivelConsistencia,DataHora,MediaDiaria,Cota01,Cota01Status,Cota24,DataIns
8,1,2010-10-01 00:00:00,1,1669.5,NaN,1363.5,2011-05-19 00:00:00
9,2,2010-10-01 00:00:00,1,1670,1,1364,2019-12-03 00:00:00
10,1,2010-10-01 07:00:00,0,1672,NaN,1363,2011-05-19 00:00:00
11,1,2010-10-01 17:00:00,0,1667,NaN,1364,2011-05-19 00:00:00


Campos que **não** passam para a tabela pré-formatada: os resumos que a ANA já calcula (`Maxima`, `Minima`, `Media`, `DiaMaxima`, `DiaMinima` e seus status), `TipoMedicaoCotas` e `DataIns` (quando o dado entrou no sistema). Eles continuam disponíveis no cache.

### 3. Pré-formatado: uma linha por dia

`ler_convencional` desdobra as 31 colunas de cada mês em linhas e converte os tipos. O valor da cota é o mesmo do XML.

In [5]:
conv = ana.ler_convencional(xml)
print(conv.shape)
print(conv.dtypes.to_string())
conv.head(5)

(1460, 6)
data            datetime64[s]
cota_cm               float64
status                  Int64
consistencia            int64
media_diaria             bool
hora                    int64


,data,cota_cm,status,consistencia,media_diaria,hora
0,2010-12-01,1645.0,<NA>,1,True,0
1,2010-12-02,1660.5,<NA>,1,True,0
2,2010-12-03,1673.0,<NA>,1,True,0
3,2010-12-04,1685.0,<NA>,1,True,0
4,2010-12-05,1699.0,<NA>,1,True,0


In [6]:
# Conferência: o dia 01/10/2010 aparece nas quatro versões, com os mesmos valores do XML
conv[conv["data"] == "2010-10-01"]

,data,cota_cm,status,consistencia,media_diaria,hora
244,2010-10-01,1669.5,<NA>,1,True,0
275,2010-10-01,1670.0,1,2,True,0
306,2010-10-01,1672.0,<NA>,1,False,7
337,2010-10-01,1667.0,<NA>,1,False,17


---
## Telemetria (sensor automático)

### 1. Resposta original

Pedimos um mês por vez. Exemplo: Manaus, outubro de 2023 (a vazante recorde).

In [7]:
xml_t = ana.xml_telemetria(MANAUS, 2023, 10)
texto_t = xml_t.decode("utf-8")
inicio = texto_t.index("<DadosHidrometereologicos ")
print(texto_t[inicio:inicio + 700])

<DadosHidrometereologicos diffgr:id="DadosHidrometereologicos1" msdata:rowOrder="0">
        <CodEstacao>14990000</CodEstacao>
        <DataHora>2023-10-31 23:45:00 </DataHora>
        <Vazao />
        <Nivel>1295.00</Nivel>
        <Chuva>0.00</Chuva>
      </DadosHidrometereologicos>
      <DadosHidrometereologicos diffgr:id="DadosHidrometereologicos2" msdata:rowOrder="1">
        <CodEstacao>14990000</CodEstacao>
        <DataHora>2023-10-31 23:30:00 </DataHora>
        <Vazao />
        <Nivel>1294.00</Nivel>
        <Chuva>0.00</Chuva>
      </DadosHidrometereologicos>
      <DadosHidrometereologicos diffgr:id="DadosHidrometereologicos3" msdata:rowOrder="2">
        <Cod


### 2. XML como tabela

Cada `<DadosHidrometereologicos>` é **uma leitura** (a cada 15 min). A ANA manda da mais recente para a mais antiga.

In [8]:
leituras = ana.tabela_xml(xml_t, "DadosHidrometereologicos")
print(leituras.shape, "→ (leituras, campos)")
leituras.head(5)

(2976, 5) → (leituras, campos)


,CodEstacao,DataHora,Vazao,Nivel,Chuva
0,14990000,2023-10-31 23:45:00,None,1295.00,0.00
1,14990000,2023-10-31 23:30:00,None,1294.00,0.00
2,14990000,2023-10-31 23:15:00,None,1294.00,0.00
3,14990000,2023-10-31 23:00:00,None,1294.00,0.00
4,14990000,2023-10-31 22:45:00,None,1294.00,0.00


### 3. Pré-formatado

`ler_telemetria` renomeia as colunas, converte os tipos e ordena do mais antigo para o mais recente. Em Manaus a vazão vem vazia (`NaN`).

In [9]:
tele = ana.ler_telemetria(xml_t)
print(tele.shape)
print(tele.dtypes.to_string())
tele.head(5)

(2976, 4)


datahora    datetime64[us]
nivel_cm           float64
vazao              float64
chuva              float64


,datahora,nivel_cm,vazao,chuva
0,2023-10-01 00:00:00,1549.0,NaN,0.0
1,2023-10-01 00:15:00,1548.0,NaN,0.0
2,2023-10-01 00:30:00,1547.0,NaN,0.0
3,2023-10-01 00:45:00,1548.0,NaN,0.0
4,2023-10-01 01:00:00,1547.0,NaN,0.0


---
## Porto de Manaus (régua oficial)

### 1. Resposta original

O site publica uma tabela por mês. A página de nível aceita filtro por ano na URL e mostra 6 meses por página. Exemplo: 2024, página 1 (julho a dezembro).

In [10]:
anos = porto.anos_disponiveis()  # {ano: id do filtro no site}
pagina = porto.html_ano(2024, anos[2024], 1)
texto_p = pagina.decode("utf-8")
inicio = texto_p.index("Outubro 2024")
print(texto_p[inicio:inicio + 1200])

Outubro 2024</span>
									</li>
								<li class="elementor-icon-list-item elementor-inline-item">
											<span class="elementor-icon-list-icon">
							<svg aria-hidden="true" class="e-font-icon-svg e-far-calendar-alt" viewBox="0 0 448 512" xmlns="http://www.w3.org/2000/svg"><path d="M148 288h-40c-6.6 0-12-5.4-12-12v-40c0-6.6 5.4-12 12-12h40c6.6 0 12 5.4 12 12v40c0 6.6-5.4 12-12 12zm108-12v-40c0-6.6-5.4-12-12-12h-40c-6.6 0-12 5.4-12 12v40c0 6.6 5.4 12 12 12h40c6.6 0 12-5.4 12-12zm96 0v-40c0-6.6-5.4-12-12-12h-40c-6.6 0-12 5.4-12 12v40c0 6.6 5.4 12 12 12h40c6.6 0 12-5.4 12-12zm-96 96v-40c0-6.6-5.4-12-12-12h-40c-6.6 0-12 5.4-12 12v40c0 6.6 5.4 12 12 12h40c6.6 0 12-5.4 12-12zm-96 0v-40c0-6.6-5.4-12-12-12h-40c-6.6 0-12 5.4-12 12v40c0 6.6 5.4 12 12 12h40c6.6 0 12-5.4 12-12zm192 0v-40c0-6.6-5.4-12-12-12h-40c-6.6 0-12 5.4-12 12v40c0 6.6 5.4 12 12 12h40c6.6 0 12-5.4 12-12zm96-260v352c0 26.5-21.5 48-48 48H48c-26.5 0-48-21.5-48-48V112c0-26.5 21.5-48 48-48h48V12c0-6.6 5.4-12 12-12h40c6.6 

### 2. HTML como tabela

`ler_pagina` acha cada mês (título + tabela) e guarda cada linha da tabela como foi publicada, com o número da linha. A cota ainda é texto.

In [11]:
linhas = porto.ler_pagina(pagina, 2024)
print(linhas.shape)
linhas.head(5)

(184, 8)


,ano,mes,dia_publicado,linha,cota_m_texto,variacao_texto,titulo,post_id
0,2024,12,1,1,14.41,4.00,Dezembro 2024,1492
1,2024,12,2,2,14.45,4.00,Dezembro 2024,1492
2,2024,12,3,3,14.49,4.00,Dezembro 2024,1492
3,2024,12,4,4,14.57,8.00,Dezembro 2024,1492
4,2024,12,5,5,14.65,8.00,Dezembro 2024,1492


### 3. Pré-formatado: datas corrigidas e cota em centímetros

`corrigir_datas` monta a data de cada linha e corrige os erros de digitação do site, registrando cada correção na coluna `correcao`. Depois a cota é convertida de metros (texto) para centímetros.

In [12]:
raw_dir = ana.CACHE_DIR.parents[1] / "raw"
porto_raw = pd.read_parquet(raw_dir / "porto_manaus.parquet")
print(porto_raw.shape)
print(porto_raw.dtypes.to_string())
porto_raw.head(5)

(9773, 8)
data              datetime64[us]
cota_cm                  float64
cota_m_texto                 str
variacao_texto               str
data_publicada    datetime64[us]
correcao                     str
titulo                       str
post_id                    int64


,data,cota_cm,cota_m_texto,variacao_texto,data_publicada,correcao,titulo,post_id
0,2000-01-01,1915.0,"19,15","15,00",2000-01-01,NaN,JANEIRO 2000,1437
1,2000-01-02,1931.0,"19,31","16,00",2000-01-02,NaN,JANEIRO 2000,1437
2,2000-01-03,1947.0,"19,47","16,00",2000-01-03,NaN,JANEIRO 2000,1437
3,2000-01-04,1960.0,"19,60","13,00",2000-01-04,NaN,JANEIRO 2000,1437
4,2000-01-05,1969.0,"19,69","9,00",2000-01-05,NaN,JANEIRO 2000,1437


In [13]:
# O recorde de 2024 como o porto publicou
porto_raw[porto_raw["data"].between("2024-10-07", "2024-10-11")]

,data,cota_cm,cota_m_texto,variacao_texto,data_publicada,correcao,titulo,post_id
9043,2024-10-07,1229.0,12.29,-12.00,2024-10-07,NaN,Outubro 2024,1298
9044,2024-10-08,1217.0,12.17,-12.00,2024-10-08,NaN,Outubro 2024,1298
9045,2024-10-09,1211.0,12.11,-6.00,2024-10-09,NaN,Outubro 2024,1298
9046,2024-10-10,1211.0,12.11,0.00 PARADO,2024-10-10,NaN,Outubro 2024,1298
9047,2024-10-11,1211.0,12.11,0.00,2024-10-11,NaN,Outubro 2024,1298


In [14]:
# Todas as correções de data
corrigidas = porto_raw.dropna(subset=["correcao"])
print(corrigidas["correcao"].str.split(",").str[0].value_counts().to_string())
corrigidas[corrigidas["correcao"].str.startswith("dia")][["data", "data_publicada", "cota_m_texto", "titulo", "correcao"]].head(10)

correcao
título diz 2023     30
dia publicado 19     3
dia publicado 21     2
dia publicado 20     2
dia publicado 26     2
dia publicado 18     2
dia publicado 16     1
dia publicado 5      1
dia publicado 10     1
dia publicado 22     1
dia publicado 23     1
dia publicado 24     1
dia publicado 25     1


,data,data_publicada,cota_m_texto,titulo,correcao
1880,2005-02-23,2005-02-21,"22,97",FEVEREIRO 2005,"dia publicado 21, data pela posição da linha"
2786,2007-08-19,2007-08-20,"25,94",AGOSTO 2007,"dia publicado 20, data pela posição da linha"
4368,2011-12-18,2011-12-16,"19,30",DEZEMBRO 2011,"dia publicado 16, data pela posição da linha"
5168,2014-02-27,2014-02-26,"24,60",FEVEREIRO 2014,"dia publicado 26, data pela posição da linha"
6911,2018-12-06,2018-12-05,"19,33",DEZEMBRO 2018,"dia publicado 5, data pela posição da linha"
8439,2023-02-11,2023-02-10,"22,81",FEVEREIRO 2023,"dia publicado 10, data pela posição da linha"
8475,2023-03-19,2023-03-18,"25,58",MARÇO 2023,"dia publicado 18, data pela posição da linha"
8718,2023-11-17,2023-11-18,"12,96",NOVEMBRO 2023,"dia publicado 18, data pela posição da linha"
8719,2023-11-18,2023-11-19,"12,98",NOVEMBRO 2023,"dia publicado 19, data pela posição da linha"
8720,2023-11-19,2023-11-20,"13,03",NOVEMBRO 2023,"dia publicado 20, data pela posição da linha"


---
## As tabelas finais em `data/raw/`

O download (`uv run python -m supplyrisk.download`) repete as etapas acima para todos os anos/meses e junta tudo em um arquivo por estação e fonte. Na telemetria, leituras repetidas (mesmo horário) aparecem uma vez só.

In [15]:
for arquivo in sorted(ana.CACHE_DIR.parents[1].joinpath("raw").glob("*.parquet")):
    df = pd.read_parquet(arquivo)
    print(f"\n=== {arquivo.name}: {len(df):,} linhas ===")
    display(df.head(5))


=== ana_convencional_14100000.parquet: 63,941 linhas ===


,data,cota_cm,status,consistencia,media_diaria,hora
0,1972-12-01,1102.0,1,1,True,0
1,1972-12-02,1109.0,1,1,True,0
2,1972-12-03,1116.0,1,1,True,0
3,1972-12-04,1121.0,1,1,True,0
4,1972-12-05,1125.0,1,1,True,0



=== ana_convencional_14990000.parquet: 86,825 linhas ===


,data,cota_cm,status,consistencia,media_diaria,hora
0,1902-12-01,1683.0,1,1,True,0
1,1902-12-02,1684.0,1,1,True,0
2,1902-12-03,1689.0,1,1,True,0
3,1902-12-04,1694.0,1,1,True,0
4,1902-12-05,1699.0,1,1,True,0



=== ana_telemetria_14100000.parquet: 510,282 linhas ===


,datahora,nivel_cm,vazao,chuva
0,2008-01-01 00:00:00,NaN,NaN,0.0
1,2008-01-01 01:00:00,1226.0,82996.8,0.0
2,2008-01-01 02:00:00,1226.0,82996.8,0.0
3,2008-01-01 03:00:00,1226.0,82996.8,0.0
4,2008-01-01 04:00:00,NaN,NaN,1.0



=== ana_telemetria_14990000.parquet: 447,603 linhas ===


,datahora,nivel_cm,vazao,chuva
0,2009-01-01 07:00:00,2243.0,NaN,NaN
1,2009-01-02 07:00:00,2254.0,NaN,NaN
2,2009-01-03 07:00:00,2263.0,NaN,NaN
3,2009-01-04 07:00:00,2272.0,NaN,NaN
4,2009-01-05 07:00:00,2282.0,NaN,NaN



=== porto_manaus.parquet: 9,773 linhas ===


,data,cota_cm,cota_m_texto,variacao_texto,data_publicada,correcao,titulo,post_id
0,2000-01-01,1915.0,"19,15","15,00",2000-01-01,NaN,JANEIRO 2000,1437
1,2000-01-02,1931.0,"19,31","16,00",2000-01-02,NaN,JANEIRO 2000,1437
2,2000-01-03,1947.0,"19,47","16,00",2000-01-03,NaN,JANEIRO 2000,1437
3,2000-01-04,1960.0,"19,60","13,00",2000-01-04,NaN,JANEIRO 2000,1437
4,2000-01-05,1969.0,"19,69","9,00",2000-01-05,NaN,JANEIRO 2000,1437


### Marcas de qualidade (`status`)

Cada valor diário da convencional traz um código de qualidade. Convenção do HidroWeb (**a confirmar com a ANA**): 0 = em branco, 1 = real, 2 = estimado, 3 = duvidoso, 4 = régua seca. Muitos dados brutos vêm sem status (`<NA>`).

In [16]:
raw_dir = ana.CACHE_DIR.parents[1] / "raw"
for codigo in (MANAUS, MANACAPURU):
    arquivo = raw_dir / f"ana_convencional_{codigo}.parquet"
    if arquivo.exists():
        df = pd.read_parquet(arquivo)
        print(f"\n{codigo}")
        print(df.groupby(["consistencia", "media_diaria", "status"], dropna=False).size().to_string())


14990000
consistencia  media_diaria  status
1             False         1          3315
                            2            59
                            <NA>       1610
              True          0           471
                            1         39504
                            2            59
                            <NA>        798
2             True          1         40758
                            2           251

14100000
consistencia  media_diaria  status
1             False         1         15621
                            2           215
                            4            28
                            <NA>      10165
              True          0           490
                            1         14147
                            2           432
                            3            14
                            <NA>       3890
2             True          1         18241
                            2           698
